# BMS Validation Lab -- Analysis Notebook

This is a scaffold. Fill in the path to your campaign's `joined.parquet` and work through each unknown section. Each section starts with the most likely PACE hypothesis and falls through to general analysis if the PACE hypothesis does not fit.

This notebook is the analyst's starting point for a validation-lab campaign capture (see `docs/06-wire-captures.md` for the methodology).

## Setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CAPTURE = Path('path/to/your/capture/directory')  # set this to your captures directory before running
df = pd.read_parquet(CAPTURE / 'joined.parquet')
print(df.shape, df.columns.tolist())


## PACE reference

In [ ]:
import sys
sys.path.insert(0, str(Path('.').resolve().parent))
from tools.pace_reference import (
    PACK_ALARM_BITS, PACK_STATUS_BITS, PROTECTION_FIELD_NAMES,
)
PACK_ALARM_BITS


## Timeline overview

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axes[0].plot(df['ts'], df.get('tcp_battery_soc', pd.Series([], dtype=float)), label='TCP SoC')
axes[0].set_ylabel('SoC %')
axes[1].plot(df['ts'], df.get('hr23_pack_current_dA', pd.Series([], dtype=float)) / 10, label='Pack current (A)')
axes[1].set_ylabel('A')
axes[2].plot(df['ts'], df.get('pack_voltage_mV', pd.Series([], dtype=float)) / 1000, label='Pack V')
axes[2].set_ylabel('V')
for ax in axes:
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
plt.tight_layout()


## A. Reg 11 (`SoC * 100`) -- what does it actually track?

**PACE hypothesis (test first):** main pack SoC, scaled by 100. PACE clients compute SoC as `REMAIN_CAP / TOTAL_CAP * 100`; reg 11 might be that value times 100.

**Fall-through:** if reg 11 does not track main SoC, look for per-cell SoC, cycle-life percent, or a stale field.


In [ ]:
# PACE hypothesis: reg 11 == main pack SoC * 100
if 'tcp_battery_soc' in df.columns and 'hr11_soc_x100' in df.columns:
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.scatter(df['tcp_battery_soc'] * 100, df['hr11_soc_x100'], s=4, alpha=0.4)
    lim = [0, 10000]; ax.plot(lim, lim, 'r--', alpha=0.5)
    ax.set_xlabel('TCP SoC * 100'); ax.set_ylabel('HR reg 11')
    ax.set_title('PACE hypothesis: reg 11 == main pack SoC * 100')
else:
    print('Required columns missing; cannot evaluate PACE hypothesis')


In [ ]:
# Fall-through: characterise reg 11's variation if it does not track main SoC.
# Look at correlation against pack current, individual cell voltages, etc.
df[['hr11_soc_x100']].describe()


## B. Reg 19 status flags -- which PACE alarm bit is which?

**PACE hypothesis (test first):** reg 19's 8 bits map 1:1 to PACE `CID2=0x44 GetAlarmInfo` pack-alarm-byte bit positions (see `PACK_ALARM_BITS`).

Find scenarios where each TCP-reported state changes (charge, discharge, balancing, low-voltage protection, etc.) and check which reg 19 bits transition at the same moment.


In [ ]:
# Per-bit transition map: for each bit position, find the rows where it changes.
if 'hr19_status' in df.columns:
    bits = pd.DataFrame({
        f'bit_{i}': (df['hr19_status'] >> i) & 1 for i in range(8)
    })
    bits['ts'] = df['ts'].values
    transitions = {}
    for i in range(8):
        col = f'bit_{i}'
        diff = bits[col].diff().abs() > 0
        transitions[i] = bits.loc[diff, ['ts', col]]
        print(f'bit {i} ({PACK_ALARM_BITS.get(i, "?")}): {len(transitions[i])} transitions')
else:
    print('reg 19 not present in this campaign')


## C. Block 3 bytes 32-35 -- static threshold or dynamic tracker?

**PACE hypothesis (test first):** these are per-cell over-/under-voltage protection thresholds from PACE `CID2=0x47 ChargeDischargeManagementInfo`. If so they should be near-static across all scenarios.

**Fall-through:** if they vary, check whether they track max/min cell voltage (extreme-cell tracker) or some other dynamic value.


In [ ]:
if {'block3_b32_offset', 'block3_b34_offset'}.issubset(df.columns):
    print('block3_b32 stats:'); print(df['block3_b32_offset'].describe())
    print('\nblock3_b34 stats:'); print(df['block3_b34_offset'].describe())
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(df['ts'], df['block3_b32_offset'], label='b32 (mV)')
    ax.plot(df['ts'], df['block3_b34_offset'], label='b34 (mV)')
    ax.set_ylabel('mV'); ax.legend(); ax.grid(True, alpha=0.3)
else:
    print('Block 3 unknown bytes not present')


## D. Reg 17 dynamics -- counter, hash, or something else?

**PACE hypothesis (test first):** reg 17 is a packet sequence counter -- increments monotonically each request. If so, `df['hr17_dynamic'].diff()` should be mostly +1 (modulo 65536).

**Fall-through:** check entropy and correlation against pack current, voltage, and per-cell readings.


In [ ]:
if 'hr17_dynamic' in df.columns:
    diffs = df['hr17_dynamic'].diff().value_counts().head(10)
    print('Top 10 reg-17 inter-frame deltas:'); print(diffs)
    print('\nUnique values:', df['hr17_dynamic'].nunique())
else:
    print('reg 17 not present')


## Findings summary

Fill in below as conclusions emerge. Then export to `<capture>/findings.md` (un-redacted) and run `tools/redact.py` to produce the public version that goes in the PR to `kenbell/giv-bms-analysis`.


**A. Reg 11:** _to be filled in_

**B. Reg 19 bits:** _to be filled in (per-bit table)_

**C. Block 3 bytes 32-35:** _to be filled in_

**D. Reg 17:** _to be filled in_
